<h1 align="center">Práctica 2. Búsqueda aproximada de una cadena</h1>
<h3 style="display:block; margin-top:5px;" align="center">Algorítmica</h3>
<h3 style="display:block; margin-top:5px;" align="center">2026-2027</h3>    
<h3 style="display:block; margin-top:5px;" align="center">Universitat Politècnica de València</h3>
<br>

**Pon/poned aquí tú/vuestros nombre(s):**
- Sergio Quintanilla Graciá
- Cristina Méndez Miró

## Índice
1. ### [Descripción del problema](#descripcion)
1. ### [Actividad 1: Recuperar la secuencia de edición](#recuperar_camino)
1. ### [Actividad 2: Reducción del coste espacial](#coste_espacial)
1. ### [Actividad 3: Añadir un parámetro umbral](#parada_umbral)
1. ### [Actividad 4 (Opcional): Cota optimista sencilla](#cota_longitudes)

<a id='descripcion'></a>

# Búsqueda aproximada de una cadena

El objetivo de esta práctica es recuperar el conjunto de palabras de un diccionario que sean parecidas a una palabra introducida por el usuario.

Este problema tiene muchas aplicaciones en tareas de recuperación de información cuando las palabras tienen errores ortográficos.

## Distancias de edición

Las operaciones de edición consideradas para calcular la distancia de Levenshtein entre $x$ e $y$ son:

- Insertar un carácter (denotado $\lambda \rightarrow y_j$)
- Borrar un carácter (denotado $x_i \rightarrow \lambda$)
- Sustituir un carácter por otro (denotado $x_i \rightarrow y_j$)

Aunque hemos definido la distancia de edición como el número de
operaciones de edición, es posible considerar el caso **ponderado**
donde las operaciones de edición pueden tener costes diferentes. Por
ejemplo, tiene sentido que sustituir una consonante por otra diferente
tenga un coste mayor a sustituir una vocal por la misma vocal
acentuada. En el caso ponderado el coste de cada operación vendría dado por:

- $\gamma(\lambda \rightarrow y_j)$
- $\gamma(x_i \rightarrow \lambda)$
- $\gamma(x_i \rightarrow y_j)$

En esta práctica vamos a limitarnos al caso **no ponderado**. Es decir, aunque se pueda contemplar que algunas operaciones de distancia de edición tienen un coste mayor que otras, el coste de todas las inserciones o borrados es el mismo sin importar el símbolo del alfabeto y todas las operaciones de sustitución dependen únicamente de si el símbolo que se sustituye es el mismo (un acierto, de coste cero) o es diferente (coste mayor que cero).

- $\gamma(\lambda \rightarrow y_j) = 1$
- $\gamma(x_i \rightarrow \lambda) = 1$
- $\gamma(x_i \rightarrow y_j) = \begin{cases} 
  0 & \mbox{si}\enspace x_i = y_j \\
  1 & \mbox{si}\enspace x_i \neq y_j \\
  \end{cases}$

En la siguiente ecuación recursiva $D(i,j)$ denota el coste de convertir el prefijo de longitud $i$ de $x$ en el prefijo de longitud $j$ de $y$:

\begin{equation*}
D(i,j) = \min \begin{cases}
    0 & \mbox{si}\enspace i=0 \; y \; j=0 \\
    D(i-1,j) + 1 & \mbox{si}\enspace i>0 \\
    D(i,j-1) + 1 & \mbox{si}\enspace j>0 \\
    D(i-1,j-1)  & \mbox{si}\enspace i>0, j>0, x_i = y_j \\
    D(i-1,j-1) +1 & \mbox{si}\enspace i>0, j>0, x_i \neq y_j \\
\end{cases}
\end{equation*}

De manera que $d(x,y) = D(|x|,|y|)$.



<a id='recuperar_camino'></a>
## Actividad 1: Recuperar la secuencia de edición

A continuación se proporciona la función que calcula la distancia de Levenshtein entre dos cadenas, en este caso hemos utilizado una matriz `numpy` de enteros para guardar las distancias:

In [1]:
import numpy as np
from collections.abc import Callable

def levenshtein(x: str, y: str) -> int:
    lenX, lenY = len(x), len(y)
    D = np.zeros((lenX+1, lenY+1), dtype=np.int32)
    for i in range(1, lenX+1):
        D[i, 0] = D[i-1, 0] + 1
    for j in range(1, lenY+1):
        D[0, j] = D[0, j-1] + 1
        for i in range(1, lenX+1):
            D[i, j] = min(D[i-1, j] + 1,
                          D[i, j-1] + 1,
                          D[i-1, j-1] + (x[i-1] != y[j-1]))
    return int(D[lenX,lenY]) # casting a int pq numpy usa np.int32

El objetivo en esta primera actividad es crear una versión de esta función que nos devuelva la secuencia de operaciones de edición para convertir la cadena `x` en la cadena `y`:

In [2]:
def levenshtein_edicion(
    x: str,
    y: str,
) -> tuple[int, list[tuple[str, str]]]:
    lenX, lenY = len(x), len(y)
    D = np.zeros((lenX+1, lenY+1), dtype=np.int32)
    # COMPLETAR AQUÍ (copia el código de arriba y adáptalo)
    for i in range(1, lenX+1):
        D[i, 0] = D[i-1, 0] + 1
    for j in range(1, lenY+1):
        D[0, j] = D[0, j-1] + 1
        for i in range(1, lenX+1):
            D[i, j] = min(D[i-1, j] + 1,
                          D[i, j-1] + 1,
                          D[i-1, j-1] + (x[i-1] != y[j-1]))

    secuencia = []
    i, j = lenX, lenY

    while i > 0 and j > 0:
        # True si las letras son las mismas
        if x[i - 1] == y[j - 1] and D[i, j] == D[i - 1, j - 1]:
            secuencia.insert(0, (x[i - 1], y[j - 1]))
            i -= 1
            j -= 1

        # True si ha habido una sustitución
        elif D[i, j] == D[i - 1, j - 1] + 1:
            secuencia.insert(0, (x[i - 1], y[j - 1]))
            i -= 1
            j -= 1

        # True si ha habido un borrado
        elif D[i, j] == D[i - 1, j] + 1:
            secuencia.insert(0, (x[i - 1], ''))
            i -= 1

        # Ha sido una inserción
        else:
            secuencia.insert(0, ('', y[j - 1]))
            j -= 1

    while i > 0:
        secuencia.insert(0, (x[i - 1], ''))
        i -= 1
    while j > 0:
        secuencia.insert(0, ('', y[j - 1]))
        j -= 1

    return int(D[lenX, lenY]), secuencia

La secuencia devuelta debe estar en el formato utilizado por la siguiente función que convierte una cadena en otra utilizando esa secuencia de operaciones. Aquí tienes un ejemplo de cómo convertir la cadena `'ejemplo'` en la cadena `'campos'` *(es el mismo ejemplo del libro de los apuntes de Andrés Marzal, María José Castro y Pablo Aibar):*

```python
aplicar_edicion("ejemplo",
                [('e','c'),('j','a'),('e',''),('m','m'),
                 ('p','p'),('l',''),('o','o'),('','s')])
```

La edición es una lista de tuplas, cada tupla son 2 letras donde una de ellas puede ser la cadena vacía si se trata de una inserción o de un borrado. Los aciertos también aparecen explícitamente en la secuencia.


In [3]:
def aplicar_edicion(
    original: str,
    edicion: list[tuple[str, str]]
) -> tuple[bool, int, str]:
    # devuelve True, distancia, nueva si todo ok
    # devuelve False, 0, '' si hay problemas
    cadena = list(original)
    errores = 0
    poscadena = 0
    for orig,nueva in edicion:
        print(" ".join(cadena[:poscadena] + ['<>'] + cadena[poscadena:]))
        if orig == '': # una insercion:
            cadena.insert(poscadena, nueva)
            poscadena += 1
            errores += 1
        elif orig != cadena[poscadena]:
            return False, 0, ''
        elif nueva == orig: # acierto
            poscadena += 1
        elif nueva == '': # borrado
            del cadena[poscadena]
            errores += 1
        else: # sustitucion
            cadena[poscadena] = nueva
            errores += 1
            poscadena += 1
    print(" ".join(cadena[:poscadena] + ['<>'] + cadena[poscadena:]))        
    return True,errores,''.join(cadena)


Puedes probar el siguiente ejemplo para ver que funciona:

In [4]:
aplicar_edicion("ejemplo",
                [('e','c'),('j','a'),('e',''),('m','m'),
                 ('p','p'),('l',''),('o','o'),('','s')])

<> e j e m p l o
c <> j e m p l o
c a <> e m p l o
c a <> m p l o
c a m <> p l o
c a m p <> l o
c a m p <> o
c a m p o <>
c a m p o s <>


(True, 5, 'campos')

El siguiente código lanza tu función sobre una batería de tests:

In [5]:
bateria_test = [("ejemplo","campos"),
                ("algortimac","algoritmica"),
                ("zapato","patos"),
                ("camarero","camionero")]

for original, nueva in bateria_test:
    distancia, secuencia = levenshtein_edicion(original, nueva)
    print(secuencia)
    ok, distprima, reconstruida = aplicar_edicion(original, secuencia)
    if distancia == distprima and nueva == reconstruida:
        print(f"{original} {nueva} {distancia}")

[('e', ''), ('j', 'c'), ('e', 'a'), ('m', 'm'), ('p', 'p'), ('l', 'o'), ('o', 's')]
<> e j e m p l o
<> j e m p l o
c <> e m p l o
c a <> m p l o
c a m <> p l o
c a m p <> l o
c a m p o <> o
c a m p o s <>
ejemplo campos 5
[('a', 'a'), ('l', 'l'), ('g', 'g'), ('o', 'o'), ('r', 'r'), ('', 'i'), ('t', 't'), ('', 'm'), ('i', 'i'), ('m', 'c'), ('a', 'a'), ('c', '')]
<> a l g o r t i m a c
a <> l g o r t i m a c
a l <> g o r t i m a c
a l g <> o r t i m a c
a l g o <> r t i m a c
a l g o r <> t i m a c
a l g o r i <> t i m a c
a l g o r i t <> i m a c
a l g o r i t m <> i m a c
a l g o r i t m i <> m a c
a l g o r i t m i c <> a c
a l g o r i t m i c a <> c
a l g o r i t m i c a <>
algortimac algoritmica 4
[('z', ''), ('a', ''), ('p', 'p'), ('a', 'a'), ('t', 't'), ('o', 'o'), ('', 's')]
<> z a p a t o
<> a p a t o
<> p a t o
p <> a t o
p a <> t o
p a t <> o
p a t o <>
p a t o s <>
zapato patos 3
[('c', 'c'), ('a', 'a'), ('m', 'm'), ('', 'i'), ('a', 'o'), ('r', 'n'), ('e', 'e'), ('r', 'r'), 

<a id='coste_espacial'></a>
## Actividad 2: Reducción del coste espacial

El objetivo de la siguiente actividad es reducir el coste espacial del algoritmo. Para ello, en lugar de una matriz debes utilizar dos vectores para representar la columna actual y la columna anterior. Se sugiere utilizar vectores `numpy`. Puedes inspirarte en el código de Levenshtein de arriba sabiendo que a `np.zeros` le puedes pasar el tamaño del vector:

```python
>>> import numpy as np
>>> np.zeros(10,dtype=np.int32)
array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
```

In [ ]:
def levenshtein_reduccion(
    x: str,
    y: str,
    umbral: int | None = None,
) -> int:
    #
    # esta versión no utiliza el parámetro umbral
    # aparece por compatibilidad con el siguiente ejercicio
    #
    lenX, lenY = len(x), len(y)
    Dprev    = np.zeros(lenX + 1, dtype=np.int32)
    Dcurrent = np.zeros(lenX + 1, dtype=np.int32)

    # COMPLETAR, ignora en esta función el argumento threshold
    # Fill the previous column with numbers from 0 to lenX
    for i in range(1, lenX + 1):
        Dprev[i] = i
    
    for j in range(1, lenY + 1):
        Dcurrent[0] = Dprev[0] + 1

        for i in range(1, lenX + 1):
            Dcurrent[i] = min(Dcurrent[i-1] + 1,
                              Dprev[i] + 1,
                              Dprev[i-1] + (x[i-1] != y[j-1]))

        Dprev, Dcurrent = Dcurrent, Dprev

    return int(Dprev[lenX])

In [7]:
print([levenshtein(x,y) for x,y in bateria_test])
print([levenshtein_reduccion(x,y) for x,y in bateria_test])

[5, 4, 3, 3]
[5, 4, 3, 3]


In [8]:
# comprobación de que funciona igual que la anterior:
# debería devolver True
all(levenshtein(x,y) == levenshtein_reduccion(x,y) for x,y in bateria_test)

True

<a id='parada_umbral'></a>
## Actividad 3: Añadir un parámetro umbral

Ahora vamos a cargar un vocabulario y buscar las palabras que estén a una distancia menor o igual a un parámetro `umbral` de una palabra escrita por el usuario.

Para ello primero debemos contruir un vocabulario, utilizaremos todas las palabras que aparezcan en un fragmento de *El Quijote*:

In [9]:
import re

vocab_file_path = "quijote.txt"

with open(vocab_file_path, "r", encoding='utf-8') as fr:
    # list pq queremos una lista, set para eliminar repetidos
    vocab = list(set(re.findall(r'\w+', fr.read().lower())))

print(len(vocab)) # debería de dar 22942

22942


La siguiente función `recuperar_parecidas` recibe como parámetros:

- Palabra escrita por el usuario.
- Vocabulario.
- Una función que calcula la distancia de edición y que recibe:

    - Una cadena
    - La otra cadena
    - (Opcionalmente) un parámetro `umbral` que es None o un nº entero. Si el umbral es `None` debe ignorarlo o bien asumir que es infinito...
    
- Un `umbral` que aplicar a la función anterior.

La función `recuperar_parecidas` devuelve un diccionario con pares `(palabra,distancia)` para aquellas distancias menor o igual al `umbral`.

In [10]:
def recuperar_parecidas(
    palabra: str,
    vocabulario: set[str], 
    func_dist: Callable[[str, str, int | None], int],
    umbral: int
) -> dict[str, int] :
    resul: dict[str, int] = {}
    for candidata in vocabulario:
        d = func_dist(palabra, candidata, umbral)
        if d <= umbral:
            resul[candidata] = d
    return resul

In [11]:
bateria_recuperar = ['caballeor','uqijote','idaljo']

In [12]:
import time
tini = time.process_time()
# completa para recuperar las palabras a distancias <=2 y <=3 de la batería anterior

umbrales = [2, 3]

for umbral in umbrales:
    for word in bateria_recuperar:
        out = recuperar_parecidas(word, vocab, levenshtein_reduccion, umbral)
        print(umbral, word, out)

tfin = time.process_time()
print("Ha tardado", tfin-tini)

2 caballeor {'caballeros': 2, 'caballos': 2, 'caballo': 2, 'caballera': 2, 'caballero': 2, 'caballe': 2}
2 uqijote {'quijote': 2}
2 idaljo {'dalo': 2, 'hidalgo': 2}
3 caballeor {'caballeros': 2, 'caballos': 2, 'cabello': 3, 'caballo': 2, 'acaballe': 3, 'caballerote': 3, 'cabales': 3, 'cabellos': 3, 'caballera': 2, 'caballero': 2, 'caballería': 3, 'caballe': 2}
3 uqijote {'ricote': 3, 'bigote': 3, 'gigote': 3, 'vejote': 3, 'pipote': 3, 'virote': 3, 'quijote': 2, 'picote': 3, 'quijotes': 3}
3 idaljo {'ida': 3, 'daño': 3, 'cuajo': 3, 'idio': 3, 'salgo': 3, 'dado': 3, 'dejo': 3, 'calo': 3, 'dando': 3, 'dijo': 3, 'séalo': 3, 'algo': 3, 'palmo': 3, 'idóneo': 3, 'halo': 3, 'vivaldo': 3, 'calzo': 3, 'malo': 3, 'tajo': 3, 'ídolo': 3, 'galgo': 3, 'valgo': 3, 'palo': 3, 'dario': 3, 'abajo': 3, 'ido': 3, 'idas': 3, 'hidalga': 3, 'hallo': 3, 'hidalgos': 3, 'dalo': 2, 'falto': 3, 'bajo': 3, 'alto': 3, 'dolo': 3, 'salto': 3, 'salvo': 3, 'influjo': 3, 'gallo': 3, 'dale': 3, 'dalle': 3, 'hidalgo': 2, '

A continuación debes crear una versión de Levenshtein con reducción del coste espacial que devuelva `umbral+1` en cuanto pueda determinar que la programación dinámica no será capaz de devolver un valor `<=umbral`. Se aconseja realizar esa comprobación tras terminar de recorrer una columna (al final de cada etapa del bucle que va columna a columna).

In [ ]:
def levenshtein_umbral(
    x: str,
    y: str,
    umbral: int | None = None,
) -> int:
    lenX, lenY = len(x), len(y)
    if umbral is None:
        umbral = max(lenX,lenY)

    # COMPLETAR
    Dprev    = np.zeros(lenX + 1, dtype=np.int32)
    Dcurrent = np.zeros(lenX + 1, dtype=np.int32)

    # Fill the previous column with numbers from 0 to lenX
    for i in range(1, lenX + 1):
        Dprev[i] = i
    
    for j in range(1, lenY + 1):
        Dcurrent[0] = Dprev[0] + 1

        for i in range(1, lenX + 1):
            Dcurrent[i] = min(Dcurrent[i-1] + 1,
                                Dprev[i] + 1,
                                Dprev[i-1] + (x[i-1] != y[j-1]))

        if min(Dcurrent) > umbral:
            return umbral + 1

        Dprev, Dcurrent = Dcurrent, Dprev

    return min(int(Dprev[lenX]), umbral + 1)

Repite la batería de pruebas de arriba pero probando la nueva versión. Averigua la mejora midiendo el tiempo que tarda una versión y la otra.

In [14]:
tini = time.process_time()
# completa para recuperar las palabras a distancias <=2 y <=3 de la batería anterior

umbrales = [2, 3]

for umbral in umbrales:
    for word in bateria_recuperar:
        out = recuperar_parecidas(word, vocab, levenshtein_umbral, umbral)
        print(umbral, word, out)
        
tfin = time.process_time()
print("Ha tardado", tfin-tini)

2 caballeor {'caballeros': 2, 'caballos': 2, 'caballo': 2, 'caballera': 2, 'caballero': 2, 'caballe': 2}
2 uqijote {'quijote': 2}
2 idaljo {'dalo': 2, 'hidalgo': 2}
3 caballeor {'caballeros': 2, 'caballos': 2, 'cabello': 3, 'caballo': 2, 'acaballe': 3, 'caballerote': 3, 'cabales': 3, 'cabellos': 3, 'caballera': 2, 'caballero': 2, 'caballería': 3, 'caballe': 2}
3 uqijote {'ricote': 3, 'bigote': 3, 'gigote': 3, 'vejote': 3, 'pipote': 3, 'virote': 3, 'quijote': 2, 'picote': 3, 'quijotes': 3}
3 idaljo {'ida': 3, 'daño': 3, 'cuajo': 3, 'idio': 3, 'salgo': 3, 'dado': 3, 'dejo': 3, 'calo': 3, 'dando': 3, 'dijo': 3, 'séalo': 3, 'algo': 3, 'palmo': 3, 'idóneo': 3, 'halo': 3, 'vivaldo': 3, 'calzo': 3, 'malo': 3, 'tajo': 3, 'ídolo': 3, 'galgo': 3, 'valgo': 3, 'palo': 3, 'dario': 3, 'abajo': 3, 'ido': 3, 'idas': 3, 'hidalga': 3, 'hallo': 3, 'hidalgos': 3, 'dalo': 2, 'falto': 3, 'bajo': 3, 'alto': 3, 'dolo': 3, 'salto': 3, 'salvo': 3, 'influjo': 3, 'gallo': 3, 'dale': 3, 'dalle': 3, 'hidalgo': 2, '

<a id='cota_longitudes'></a>
## Actividad 4: Cota optimista sencilla

Vamos a ver una cota optimista que nos va a permitir evitar el cálculo de la distancia entre dos cadenas cuando dicha cota sea mayor al umbral.

La primera cota, muy sencilla, es la diferencia en valor absoluto entre las longitudes de las dos cadenas. Razona dicha cota e impleméntala:

In [15]:
def cota_sencilla(x: str, y: str) -> int:
    # devuelve una cota optimista de la distancia de Levenshtein entre x e y
    return abs(len(x) - len(y)) # modificar

La siguiente versión de `recuperar_parecidas` hace uso de una cota para evitar algunos cálculos. Devuelve un segundo argumento que es el nº de veces que hemos evitado llamar a la función de distancia:

In [16]:
def recuperar_parecidas_cota(
    palabra: str,
    vocab: set[str],
    func_dist: Callable[[str, str, int | None], int],
    umbral: int,
    fcota: Callable[[str, str], int],
) -> tuple[dict[str, int], int]:

    resul: dict[str, int] = {}
    evitadas = 0

    for candidata in vocab:
        cota = fcota(palabra, candidata)
        if cota > umbral:
            evitadas += 1
        else:
            d = func_dist(palabra, candidata, umbral)
            if d <= umbral:
                resul[candidata] = d

    return resul, evitadas

Prueba esta versión y compárala con la anterior utilizando la cota sencilla.

In [17]:
tini = time.process_time()
# completa para recuperar las palabras a distancias <=2 y <=3 de la batería anterior

umbrales = [2, 3]

for umbral in umbrales:
    for word in bateria_recuperar:
        out, evitadas = recuperar_parecidas_cota(word, vocab, levenshtein_umbral, umbral, cota_sencilla)
        print(umbral, word, evitadas, out)
        
tfin = time.process_time()
print("Ha tardado", tfin-tini)

2 caballeor 7812 {'caballeros': 2, 'caballos': 2, 'caballo': 2, 'caballera': 2, 'caballero': 2, 'caballe': 2}
2 uqijote 6334 {'quijote': 2}
2 idaljo 8564 {'dalo': 2, 'hidalgo': 2}
3 caballeor 4002 {'caballeros': 2, 'caballos': 2, 'cabello': 3, 'caballo': 2, 'acaballe': 3, 'caballerote': 3, 'cabales': 3, 'cabellos': 3, 'caballera': 2, 'caballero': 2, 'caballería': 3, 'caballe': 2}
3 uqijote 2996 {'ricote': 3, 'bigote': 3, 'gigote': 3, 'vejote': 3, 'pipote': 3, 'virote': 3, 'quijote': 2, 'picote': 3, 'quijotes': 3}
3 idaljo 5180 {'ida': 3, 'daño': 3, 'cuajo': 3, 'idio': 3, 'salgo': 3, 'dado': 3, 'dejo': 3, 'calo': 3, 'dando': 3, 'dijo': 3, 'séalo': 3, 'algo': 3, 'palmo': 3, 'idóneo': 3, 'halo': 3, 'vivaldo': 3, 'calzo': 3, 'malo': 3, 'tajo': 3, 'ídolo': 3, 'galgo': 3, 'valgo': 3, 'palo': 3, 'dario': 3, 'abajo': 3, 'ido': 3, 'idas': 3, 'hidalga': 3, 'hallo': 3, 'hidalgos': 3, 'dalo': 2, 'falto': 3, 'bajo': 3, 'alto': 3, 'dolo': 3, 'salto': 3, 'salvo': 3, 'influjo': 3, 'gallo': 3, 'dale': 